# Optima: Colab GPU experiment

Run these cells in order, or resume at any stage when the required artifact already exists. The downstream embedding, indexing, retrieval, evaluation, and plotting code is the existing Optima implementation.

## 1. Environment setup

Enable a GPU in **Runtime > Change runtime type** before running this cell.

In [7]:
!pip install -q transformers accelerate bitsandbytes sentence-transformers tqdm matplotlib

In [8]:
import os
import sys
import platform
import subprocess
from pathlib import Path

import torch

cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")
print(f"GPU: {torch.cuda.get_device_name(0) if cuda_available else 'none'}")
if cuda_available:
    memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU memory: {memory_gb:.2f} GB")
else:
    print("WARNING: No GPU is available. Do not load a large LLM until a Colab GPU runtime is enabled.")
print(f"PyTorch version: {torch.__version__}")
print(f"Python version: {platform.python_version()}")

CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB
PyTorch version: 2.11.0+cu128
Python version: 3.13.15


## 2. Project setup

This works from a cloned repository or an uploaded notebook. The repository root is added automatically.

In [11]:
import colab.colab_pipeline as cp

functions = [
    "build_index",
    "build_raw_index",
    "enrich_nodes",
    "inspect_json",
    "load_json",
    "load_model",
    "prepare_workspace",
    "retrieve",
    "run_evaluation",
    "save_json",
    "save_experiment_summary",
    "validate_json",
    "zip_outputs",
]

for name in functions:
    print(f"{'✓' if hasattr(cp, name) else '✗'} {name}")

✓ build_index
✓ build_raw_index
✓ enrich_nodes
✓ inspect_json
✓ load_json
✓ load_model
✓ prepare_workspace
✓ retrieve
✓ run_evaluation
✓ save_json
✗ save_experiment_summary
✗ validate_json
✓ zip_outputs


In [12]:
from pathlib import Path
import sys
import subprocess

# ============================================================
# 1. LOCATE / CLONE OPTIMA
# ============================================================

candidates = [
    Path.cwd(),
    Path("/content/optima_python"),
    Path("/content/optima"),
    Path("/content"),
]

project_found = any((p / "optima").is_dir() for p in candidates)

if not project_found:
    print(
        "Repository files are not present in this Colab runtime; "
        "cloning Optima..."
    )

    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/I1gorr/optima_python.git",
            "/content/optima_python",
        ],
        check=True,
    )

    candidates.insert(0, Path("/content/optima_python"))

PROJECT_ROOT = next(
    (p for p in candidates if (p / "optima").is_dir()),
    None,
)

if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not locate the Optima repository."
    )

print(f"Project root: {PROJECT_ROOT}")


# ============================================================
# 2. PYTHON PATH
# ============================================================

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# ============================================================
# 3. INSTALL OPTIMA
# ============================================================

print("Installing Optima...")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        str(PROJECT_ROOT),
    ],
    check=True,
)


# ============================================================
# 4. IMPORT COLAB PIPELINE
# ============================================================

from colab.colab_pipeline import (
    build_index,
    build_raw_index,
    enrich_nodes,
    inspect_json,
    load_json,
    load_model,
    prepare_workspace,
    retrieve,
    run_evaluation,
    save_json,
    save_experiment_summary,
    validate_json,
    zip_outputs,
)


# ============================================================
# 5. OUTPUT DIRECTORIES
# ============================================================

OUTPUT_ROOT = PROJECT_ROOT / "colab_outputs"

INPUT_ROOT = OUTPUT_ROOT / "inputs"
ENRICHMENT_ROOT = OUTPUT_ROOT / "enrichment"
REPORT_ROOT = OUTPUT_ROOT / "retrieval"

for directory in [
    OUTPUT_ROOT,
    INPUT_ROOT,
    ENRICHMENT_ROOT,
    REPORT_ROOT,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# 6. SUMMARY
# ============================================================

print()
print("=" * 60)
print("OPTIMA COLAB ENVIRONMENT READY")
print("=" * 60)
print(f"Project root : {PROJECT_ROOT}")
print(f"Artifacts    : {OUTPUT_ROOT}")
print(f"Inputs       : {INPUT_ROOT}")
print(f"Enrichment   : {ENRICHMENT_ROOT}")
print(f"Retrieval    : {REPORT_ROOT}")
print("=" * 60)

Project root: /content/optima_python
Installing Optima...


ImportError: cannot import name 'save_experiment_summary' from 'colab.colab_pipeline' (/content/optima_python/colab/colab_pipeline.py)

## 3. Upload existing JSON artifacts

Upload `base.json`, one or more enriched JSON files, or both. The source files are copied into `colab_outputs/inputs/` and are never overwritten.

In [ ]:
from google.colab import files
import json
import shutil

uploaded = files.upload()
uploaded_paths = []
for filename in uploaded:
    source = Path(filename)
    destination = INPUT_ROOT / source.name
    shutil.copy2(source, destination)
    uploaded_paths.append(destination)
print('Uploaded files:')
for path in uploaded_paths:
    print(f'- {path.name}')

json_inputs = sorted(INPUT_ROOT.glob('*.json'))
if not json_inputs:
    raise FileNotFoundError('Upload at least one Optima JSON artifact.')
base_candidates = [p for p in json_inputs if p.name.lower() == 'base.json']
enriched_candidates = [p for p in json_inputs if 'enrich' in p.name.lower() or 'enhanced' in p.name.lower()]
INPUT_JSON = base_candidates[0] if base_candidates else (enriched_candidates[0] if enriched_candidates else json_inputs[0])
print(f'Automatically selected INPUT_JSON: {INPUT_JSON}')

## 4. Inspect the uploaded JSON

In [ ]:
inspection = validate_json(INPUT_JSON)
HAS_ENRICHMENT = not inspection['enrichment_required']
print(f'Already enriched: {HAS_ENRICHMENT}')

## 5. Configuration

Change only these values for a new experiment. Keep `EMBEDDING_MODEL`, `REPRESENTATION_MODE`, benchmark settings, and `K` fixed when comparing enrichment models.

In [ ]:
# Change MODEL_ID to test another Hugging Face causal language model.
MODEL_ID = 'ibm-granite/granite-3.3-8b-instruct'
# Change INPUT_JSON when selecting another uploaded base/enriched artifact.
# INPUT_JSON = INPUT_ROOT / 'base.json'
# Use the same alias/configuration as local Optima for controlled comparisons.
EMBEDDING_MODEL = 'bge-small'
REPRESENTATION_MODE = 'hybrid'
K = 10
NUM_QUERIES = 20
FORCE_REBUILD = False
LOAD_IN_4BIT = True
MAX_NEW_TOKENS = 768

MODEL_NAME = MODEL_ID.split('/')[-1].replace('_', '-').lower()
EXPERIMENT_ROOT = OUTPUT_ROOT / MODEL_NAME / EMBEDDING_MODEL
ENRICHED_JSON = EXPERIMENT_ROOT / 'enrichment' / 'enriched.json'
WORKSPACE = EXPERIMENT_ROOT / 'workspace'
REPORT_ROOT = EXPERIMENT_ROOT / 'retrieval'
EVALUATION_ROOT = EXPERIMENT_ROOT / 'evaluation'
PLOTS_ROOT = EVALUATION_ROOT / 'plots'
print(f'Model: {MODEL_ID}')
print(f'Embedding alias: {EMBEDDING_MODEL}')
print(f'Experiment output: {EXPERIMENT_ROOT}')

## 6. LLM enrichment (optional when an enriched artifact is uploaded)

The adapter uses the existing Optima v2 semantic prompt and schema. It writes after each node, retries malformed responses, records failures and latency, and can resume from a previous enriched JSON.

In [ ]:
ENRICHMENT_METRICS = {}
if HAS_ENRICHMENT:
    ENRICHED_JSON.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(INPUT_JSON, ENRICHED_JSON)
    print(f'Using uploaded enriched artifact; enrichment skipped: {ENRICHED_JSON}')
else:
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA is unavailable. Enable a Colab GPU before loading the configured LLM.')
    tokenizer, llm = load_model(MODEL_ID, load_in_4bit=LOAD_IN_4BIT)
    metrics = enrich_nodes(
        INPUT_JSON, ENRICHED_JSON, MODEL_ID, llm, tokenizer,
        max_new_tokens=MAX_NEW_TOKENS,
    )
    ENRICHMENT_METRICS = metrics
    print(metrics)

## 7. Embedding and corpus/index generation

This calls the existing Optima document constructor and embedding registry. Both raw and enriched corpora are indexed so the raw baseline remains available.

In [ ]:
base_json = INPUT_ROOT / 'base.json'
if not base_json.exists() and not HAS_ENRICHMENT:
    base_json = INPUT_JSON
if not base_json.exists():
    print('No base.json uploaded: raw baseline will be skipped.')

prepare_workspace(base_json if base_json.exists() else None, ENRICHED_JSON, WORKSPACE, MODEL_NAME)
raw_result = None
if base_json.exists():
    raw_result = build_raw_index(base_json, EMBEDDING_MODEL, REPORT_ROOT, REPRESENTATION_MODE, FORCE_REBUILD)
enriched_result = build_index(ENRICHED_JSON, MODEL_NAME, EMBEDDING_MODEL, REPORT_ROOT, REPRESENTATION_MODE, FORCE_REBUILD)
print('Raw index:', raw_result)
print('Enriched index:', enriched_result)
print(f"Documents created: {enriched_result.get('num_documents', 'reused index')}")
print(f"Representation mode: {REPRESENTATION_MODE}")
print(f"Fields embedded: compiler fields plus semantic enrichment where available")

## 8. Retrieval smoke test

In [ ]:
query_text = 'What does this function do?'
retrieved = retrieve(REPORT_ROOT, MODEL_NAME, EMBEDDING_MODEL, query_text, k=5)
for item in retrieved:
    print(f"{item['rank']}. {item['function_name']} ({item['function_id']})")
print(f"Corpus size: {enriched_result.get('num_documents', 'reused index')}")
print(f"Index: {enriched_result.get('index_metrics', {}).get('index_path', 'reused index')}")

## 9. Evaluation, metrics, and graphs

Supply an existing benchmark at `BENCHMARK_JSON` to resume evaluation without regenerating queries. Otherwise the existing Optima benchmark generator creates one from the workspace.

In [ ]:
BENCHMARK_JSON = next(iter(sorted(INPUT_ROOT.glob('*queries*.json'))), None)
matrix = run_evaluation(
    REPORT_ROOT,
    [EMBEDDING_MODEL],
    benchmark_path=BENCHMARK_JSON,
    benchmark_source=WORKSPACE,
    num_queries=NUM_QUERIES,
    k=K,
    output_dir=EVALUATION_ROOT,
)
for embedding, corpora in matrix.items():
    for corpus, metrics in corpora.items():
        print(embedding, corpus, 'Recall@5=', metrics.get('recall_at_5'), 'MRR=', metrics.get('mrr'))

## 10. Download artifacts

The ZIP includes enrichment JSON, FAISS indexes, benchmark, evaluation CSV/JSON files, per-query results, and plots.

In [ ]:
from google.colab import files

retrieval_path = REPORT_ROOT / 'manual_retrieval.json'
save_json({'query': query_text, 'k': 5, 'results': retrieved}, retrieval_path)
config_path = EXPERIMENT_ROOT / 'experiment_config.json'
save_json({
    'model_id': MODEL_ID, 'embedding_model': EMBEDDING_MODEL,
    'representation_mode': REPRESENTATION_MODE, 'k': K,
    'num_queries': NUM_QUERIES, 'load_in_4bit': LOAD_IN_4BIT,
    'max_new_tokens': MAX_NEW_TOKENS, 'force_rebuild': FORCE_REBUILD,
}, config_path)
summary_path = EVALUATION_ROOT / 'experiment_summary.json'
save_experiment_summary(
    summary_path,
    model_id=MODEL_ID,
    embedding_model=EMBEDDING_MODEL,
    representation_mode=REPRESENTATION_MODE,
    input_json=INPUT_JSON,
    nodes=inspection['nodes'],
    enrichment_metrics=ENRICHMENT_METRICS,
    index_results=[item for item in (raw_result, enriched_result) if item],
    matrix=matrix,
    k=K,
)
archive = zip_outputs(EXPERIMENT_ROOT, EXPERIMENT_ROOT / f'optima_experiment_{MODEL_NAME}_{EMBEDDING_MODEL}.zip')
print('=' * 60)
print('OPTIMA EXPERIMENT COMPLETE')
print('=' * 60)
print(f'LLM: {MODEL_ID}')
print(f'Embedding model: {EMBEDDING_MODEL}')
print(f'Representation: {REPRESENTATION_MODE}')
print(f'Input: {INPUT_JSON}')
print(f'Nodes: {inspection["nodes"]}')
print(f'Enriched JSON: {ENRICHED_JSON}')
for row in summary_path.exists() and json.loads(summary_path.read_text()).get('evaluation', []) or []:
    print(f"Evaluation {row['corpus']}: Recall@5={row['recall_at_5']}, MRR={row['mrr']}, latency={row['mean_latency']}")
print(f'Index: {REPORT_ROOT / "indexes"}')
print(f'Evaluation: {EVALUATION_ROOT}')
print(f'Plots: {PLOTS_ROOT}')
print(f'ZIP: {archive}')
files.download(str(archive))